In [2]:
import geopandas as gpd
from shapely.geometry import box
import subprocess
import requests

sierra_geom = box(-119.5, 35.5, -118.5, 37.5)
gdf = gpd.GeoDataFrame({"geometry": [sierra_geom]}, crs="EPSG:4326")
out_path = "../data/kml/sierra_pilot.geojson"
gdf.to_file(out_path, driver="GeoJSON")

assessment_id = "climate_impact_curves_sierra_2016_2018"
aoi_gs = f"gs://fire-biomass/{assessment_id}/perimeter.geojson"
subprocess.run(["gsutil", "cp", out_path, aoi_gs], check=True)

url = "https://europe-west2-artio-carbon.cloudfunctions.net/biomassagbd-trigger"

base_body = {
    "PROJECT_ASSESSMENT_ID": assessment_id,
    "AOI_GS": aoi_gs,
    "PIPELINE_COMMAND": "prepare-data",
    "AUTO_TRIGGER_STAGE2": True,
    "MODEL_IDS": "18693595-3,18693569-4",
    "PROJECT_ID": "artio-carbon",
    "STAGING_BUCKET": "gs://artio-vertex-staging/vertex-staging",
    "SERVICE_ACCOUNT": "prod-service-account@artio-carbon.iam.gserviceaccount.com",
}

for year, quarter in [("2016", "Q1"), ("2018", "Q4")]:
    body = {**base_body, "YEARS": [year], "QUARTERS": [quarter]}
    r = requests.post(url, json=body)
    print(year, quarter, r.status_code, r.text[:300])


Copying file://../data/kml/sierra_pilot.geojson [Content-Type=application/geo+json]...
/ [0 files][    0.0 B/  348.0 B]                                                

KeyboardInterrupt: 

/ [1 files][  348.0 B/  348.0 B]                                                
Operation completed over 1 objects/348.0 B.                                      


In [1]:
import subprocess

def check_job_state(assessment_id):
    result = subprocess.run(
        ["gcloud", "ai", "custom-jobs", "list", "--region=europe-west2", "--project=artio-carbon",
         f"--filter=displayName:biomassagbd-stage1-{assessment_id}*",
         "--format=value(displayName,state)"],
        capture_output=True, text=True,
    )
    print(result.stdout or "(no matching jobs found yet)")
    return result.stdout

def check_file_counts(assessment_id):
    result = subprocess.run(
        ["gsutil", "ls", f"gs://artio-projects/project-assessment/{assessment_id}/biomass/*/*/predictions/*/*.tif"],
        capture_output=True, text=True,
    )
    n_files = len([l for l in result.stdout.splitlines() if l.strip()])

    s2_result = subprocess.run(
        ["gsutil", "ls", f"gs://artio-projects/project-assessment/{assessment_id}/biomass/*/*/composite/s2/*.tif"],
        capture_output=True, text=True,
    )
    n_tiles = len([l for l in s2_result.stdout.splitlines() if l.strip()])

    expected = n_tiles * 2
    status = "COMPLETE" if n_files >= expected and n_tiles > 0 else f"incomplete ({n_files}/{expected})"
    print(assessment_id, f"tiles={n_tiles}, predictions={n_files}/{expected} -> {status}")

check_job_state(assessment_id)
check_file_counts(assessment_id)


NameError: name 'assessment_id' is not defined

In [4]:
assessment_id = "climate_impact_curves_sierra_2016_2018"

check_job_state(assessment_id)
check_file_counts(assessment_id)


biomassagbd-stage1-climate_impact_curves_sierra_2016_2018-2016-Q1-20261109-140540	JOB_STATE_FAILED
biomassagbd-stage1-climate_impact_curves_sierra_2016_2018-2018-Q4-20261109-140540	JOB_STATE_RUNNING

climate_impact_curves_sierra_2016_2018 tiles=7, predictions=0/14 -> incomplete (0/14)


In [5]:
import requests

payload_2017q1 = {
    "project_id": assessment_id,
    "aoi_gs": f"gs://fire-biomass/{assessment_id}/perimeter.geojson",
    "year": "2017",
    "quarter": "Q1",
    "output_bucket": "artio-projects",
    "composite_bucket": "artio-projects",
    "stage2_model_ids": ["18693595-3", "18693569-4"],
    # keep every other field identical to your working 2016-Q1 / 2018-Q4 request bodies
}

resp = requests.post(
    "https://europe-west2-artio-carbon.cloudfunctions.net/biomassagbd-trigger",
    json=payload_2017q1,
)
print(resp.status_code)
print(resp.json())


400
{'error': 'PROJECT_ASSESSMENT_ID is required (set env or send in request)'}
